# 03 - Modelos B3 y B4

B3 contrasta un árbol de decisión con un bosque aleatorio para detectar interacciones no lineales. B4 es el modelo principal de potenciación del gradiente por histogramas. Ambos conservan las partidas completas en la validación cruzada por `game_id`; `val` solo se usa para comparar modelos ya definidos con train.

La implementación requiere scikit-learn 1.7.2 porque permite pasar vigilancia externa a `HistGradientBoostingClassifier`. La versión anterior elegía filas al azar internamente, lo cual podía dividir partidas.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc, sys, time
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa la descarga anterior.'
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn import __version__ as sklearn_version
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.features import feature_columns
from ptcg_wp.b3 import b3_optuna_cv, fit_b3_candidates_val
from ptcg_wp.b4 import materialize_split, b4_optuna_cv, fit_b4_candidates_val
import pyarrow.parquet as pq
assert tuple(map(int, sklearn_version.split('.')[:2])) >= (1, 7), 'Ejecuta pip install -r requirements.txt y reinicia.'
SCENARIOS = ('A', 'B', 'C')
print('Entorno B3/B4 listo; scikit-learn:', sklearn_version)


## B3 — árbol y bosque aleatorio

El árbol y el bosque tienen hiperparámetros propios. Optuna TPE busca configuraciones por separado para AUC, log loss y Brier usando pliegues de partidas completas en `train`; después se comparan hasta tres finalistas por modelo en `val`. El espacio de búsqueda que sigue es inicial y explícito para que podamos revisarlo. Para el árbol se prueban criterio (`gini`, `entropy`, `log_loss`), profundidad (sin límite o 4, 8, 12, 20, 32), mínimo de 1 a 100 estados por hoja y poda `ccp_alpha` entre 1e-7 y 1e-2. Para el bosque se prueban 50–250 árboles, esos tres criterios, fracción de características 0.1–1.0, profundidad sin límite o 8, 12, 20, 32 y mínimo de 1–100 estados por hoja. El piloto toma una muestra de partidas completas y mide costo; `full` intenta usar todo train sujeto a la guarda de RAM.


In [ ]:
B3_MODE = 'pilot'
B3_BATCH_SIZE = 32768
B3_SEED = 2026
# El muestreo, cuando se usa, siempre conserva partidas completas.
B3_MAX_TRAIN_GAMES = 1000 if B3_MODE == 'pilot' else None
B3_TRIALS = 1 if B3_MODE == 'pilot' else 10
B3_FOLDS = 2 if B3_MODE == 'pilot' else 5
B3_MAX_FOLDS = 1 if B3_MODE == 'pilot' else None
B3_N_JOBS = 2  # Limita procesos paralelos para no multiplicar la RAM de Colab.
B3_MAX_RAM_GIB = 10.0
B3_OUTPUT_ROOT = CHECKPOINT_ROOT / 'b3' / B3_MODE
parquet_schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES_B3 = feature_columns(parquet_schema)
alcance_b3 = 'TODAS las partidas' if B3_MAX_TRAIN_GAMES is None else f'muestra de {B3_MAX_TRAIN_GAMES} partidas completas'
print(f'B3 | modo: {B3_MODE} | train: {alcance_b3} | características: {len(FEATURES_B3)} | folds: {B3_FOLDS} | pruebas por métrica/modelo: {B3_TRIALS}')


In [ ]:
escenarios_b3 = ('A',) if B3_MODE == 'pilot' else SCENARIOS
b3_cv_partes, b3_candidatas_partes, b3_val_partes = [], [], []
for escenario in escenarios_b3:
    salida = B3_OUTPUT_ROOT / f'escenario_{escenario}'
    stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES_B3), batch_size=B3_BATCH_SIZE)
    t0 = time.perf_counter()
    cv, candidatas = b3_optuna_cv(stream, FEATURES_B3, n_splits=B3_FOLDS, n_trials=B3_TRIALS,
                                  max_folds=B3_MAX_FOLDS, max_train_games=B3_MAX_TRAIN_GAMES,
                                  seed=B3_SEED, n_jobs=B3_N_JOBS, max_ram_gib=B3_MAX_RAM_GIB,
                                  cache_dir='/content', output_dir=salida, verbose=False)
    cv['escenario'] = escenario
    candidatas['escenario'] = escenario
    b3_cv_partes.append(cv)
    b3_candidatas_partes.append(candidatas)
    # El piloto mide costo. En full, val compara las configuraciones finalistas.
    if B3_MODE == 'full':
        b3_val_partes.append(fit_b3_candidates_val(
            stream, FEATURES_B3, candidatas, max_train_games=B3_MAX_TRAIN_GAMES, seed=B3_SEED,
            n_jobs=B3_N_JOBS, max_ram_gib=B3_MAX_RAM_GIB, cache_dir='/content', output_dir=salida, verbose=True))
    print(f'B3 {escenario}: {time.perf_counter() - t0:.1f} s')
    del stream, cv, candidatas
    gc.collect()
b3_cv = pd.concat(b3_cv_partes, ignore_index=True)
b3_candidatas = pd.concat(b3_candidatas_partes, ignore_index=True)
display(b3_candidatas.round(5))
display(b3_cv.loc[b3_cv.etapa.eq('global')].groupby(['escenario', 'modelo', 'estudio']).agg(
    auc=('auc', 'mean'), log_loss=('log_loss', 'mean'), brier=('brier', 'mean'),
    partidas=('partidas_train_muestra', 'first'), pliegues=('fold', 'count')).round(4))
B3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
b3_cv.to_csv(B3_OUTPUT_ROOT / 'b3_cv_completo.csv', index=False)
b3_candidatas.to_csv(B3_OUTPUT_ROOT / 'b3_candidatas.csv', index=False)
if b3_val_partes:
    b3_val = pd.concat(b3_val_partes, ignore_index=True)
    display(b3_val.round(4))
    b3_val.to_csv(B3_OUTPUT_ROOT / 'b3_val_final.csv', index=False)


In [ ]:
# Pilot usa dos pliegues y una prueba por métrica para medir recursos con todo train.
B4_MODE = 'pilot'
B4_BATCH_SIZE = 32768
B4_SEED = 2026
B4_MONITOR_FRACTION = 0.15
B4_MAX_ITER = 500
B4_PATIENCE = 30
B4_MAX_RAM_GIB = 10.0
B4_TRIALS = 1 if B4_MODE == 'pilot' else 10
B4_FOLDS = 2 if B4_MODE == 'pilot' else 5
B4_TIMEOUT_PER_STUDY = None  # Después del piloto se fija un presupuesto de tiempo justificado.
B4_OUTPUT_ROOT = CHECKPOINT_ROOT / 'b4' / B4_MODE
parquet_schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES_B4 = feature_columns(parquet_schema)
print('B4 | modo:', B4_MODE, '| train: TODAS las partidas | características:', len(FEATURES_B4),
      '| pliegues:', B4_FOLDS, '| pruebas por métrica:', B4_TRIALS, '| RAM máxima estimada:', B4_MAX_RAM_GIB, 'GiB')


In [ ]:
escenarios = ('A',) if B4_MODE == 'pilot' else SCENARIOS
cv_partes, candidatas_partes, val_partes = [], [], []
for escenario in escenarios:
    salida = B4_OUTPUT_ROOT / f'escenario_{escenario}'
    t0 = time.perf_counter()
    with TemporaryDirectory(prefix=f'b4-{escenario}-', dir='/content') as cache:
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES_B4), batch_size=B4_BATCH_SIZE)
        train = materialize_split(stream, FEATURES_B4, 'train', cache)
        val = materialize_split(stream, FEATURES_B4, 'val', cache)
        cv, candidatas = b4_optuna_cv(
            train, n_splits=B4_FOLDS, n_trials=B4_TRIALS, timeout=B4_TIMEOUT_PER_STUDY,
            seed=B4_SEED, monitor_fraction=B4_MONITOR_FRACTION, max_iter=B4_MAX_ITER,
            patience=B4_PATIENCE, max_ram_gib=B4_MAX_RAM_GIB, output_dir=salida, verbose=False)
        cv['escenario'] = escenario
        candidatas['escenario'] = escenario
        cv_partes.append(cv)
        candidatas_partes.append(candidatas)
        if B4_MODE == 'full':
            val_partes.append(fit_b4_candidates_val(
                train, val, candidatas, seed=B4_SEED, monitor_fraction=B4_MONITOR_FRACTION,
                max_iter=B4_MAX_ITER, patience=B4_PATIENCE, max_ram_gib=B4_MAX_RAM_GIB,
                output_dir=salida, scenario=escenario, verbose=False))
        train.close(); val.close()
    print(f'B4 {escenario}: {time.perf_counter() - t0:.1f} s; candidatas={len(candidatas)}')
    del stream, train, val, cv, candidatas
    gc.collect()
b4_cv = pd.concat(cv_partes, ignore_index=True)
b4_candidates = pd.concat(candidatas_partes, ignore_index=True)
display(b4_candidates.round(5))
display(b4_cv.loc[b4_cv.etapa.eq('global')].groupby(['escenario', 'estudio']).agg(
    auc=('auc', 'mean'), log_loss=('log_loss', 'mean'), brier=('brier', 'mean'),
    n_iter=('n_iter', 'mean'), segundos=('tiempo_segundos', 'max')).round(4))
b4_cv.to_csv(B4_OUTPUT_ROOT / 'b4_cv_completo.csv', index=False)
b4_candidates.to_csv(B4_OUTPUT_ROOT / 'b4_candidatas.csv', index=False)
if val_partes:
    b4_val = pd.concat(val_partes, ignore_index=True)
    display(b4_val.round(4))
    b4_val.to_csv(B4_OUTPUT_ROOT / 'b4_val_finalistas.csv', index=False)


## Leer el piloto

El piloto usa todos los juegos disponibles de `train` en A, pero dos pliegues y una prueba TPE por métrica. Sirve para medir tiempo, RAM y espacio local; no sirve para elegir configuración. En `full`, cada una de A, B y C ejecuta cinco pliegues y el presupuesto definido de pruebas por métrica.

Cada modelo de un pliegue se ajusta con partidas de modelo, observa partidas diferentes de vigilancia para la parada temprana y se evalúa sobre el pliegue externo. Por eso `partidas_modelo`, `partidas_vigilancia` y `partidas_evaluacion` no se superponen. Después, cada finalista usa vigilancia solo para decidir cuántos árboles usar y se reentrena con todas las partidas de train antes de evaluar `val`. `calib` y `test` no se leen.

La búsqueda TPE se ejecuta una vez por AUC, log loss y Brier para no fusionar las tres métricas con pesos arbitrarios. Las hasta tres candidatas se comparan en `val` siguiendo la regla acordada. Los estudios Optuna y los CSV se guardan en `MyDrive/Applied-AI-Agent-Project/model_checkpoints`; al reiniciar Colab, vuelve a ejecutar el cuaderno con el mismo modo y configuración para reanudar. Los trials ya completados se conservan; si se interrumpe un trial a mitad, ese trial se repite desde el inicio. La evaluación finalista también guarda progreso por candidata. La materialización temporal de B4 se reconstruye localmente en cada runtime.
